In [ ]:
from astropy.config import generate_config
generate_config(pkgname='stsynphot')

from stsynphot import conf

conf.rootdir = '/my/local/dir/trds'
conf.graphtable = '/my/local/dir/trds/mtab/07r1502mm_tmg.fits'
conf.comptable = '/my/local/dir/trds/mtab/07r1502nm_tmc.fits'
conf.thermtable = '/my/local/dir/trds/mtab/tae17277m_tmt.fits'
conf.area = 331830.72404

In [ ]:
import stsynphot as stsyn
stsyn.showref()


INFO: 
graphtable: 
comptable : 
thermtable: 
area      : 45238.93416
waveset   : Min: 500, Max: 26000, Num: 10000, Delta: None, Log: True
 [stsynphot.config]


AttributeError: 'NoneType' object has no attribute 'plot'

In [6]:
from stsynphot import conf
print(conf.rootdir)

/grp/redcat/trds


In [7]:
import os
import astropy.units as u
from astropy.table import Table
from astroquery.svo_fps import SvoFps

# List all NIRCam filters
flt = SvoFps.get_filter_list('JWST', instrument='NIRCam')
print(flt['filterID'])

# Save every curve to a local folder
os.makedirs('nircam_throughput', exist_ok=True)
for fid in flt['filterID']:
    tbl = SvoFps.get_transmission_data(fid)
    name = fid.split('.')[-1]
    tbl.write(f'nircam_throughput/{name}.ecsv', overwrite=True)

       filterID      
                     
---------------------
    JWST/NIRCam.F070W
JWST/NIRCam2025.F070W
    JWST/NIRCam.F090W
JWST/NIRCam2025.F090W
JWST/NIRCam2025.F115W
    JWST/NIRCam.F115W
    JWST/NIRCam.F140M
JWST/NIRCam2025.F140M
    JWST/NIRCam.F150W
                  ...
JWST/NIRCam2025.F444W
    JWST/NIRCam.F444W
JWST/NIRCam2025.F460M
    JWST/NIRCam.F460M
JWST/NIRCam2025.F466N
    JWST/NIRCam.F466N
    JWST/NIRCam.F470N
JWST/NIRCam2025.F470N
JWST/NIRCam2025.F480M
    JWST/NIRCam.F480M
Length = 58 rows


In [13]:

import os
os.environ['PYSYN_CDBS'] = os.path.expanduser('~/trds')
import stsynphot as stsyn
import numpy as np
import astropy.units as u
from astropy.table import Table
from synphot import SpectralElement, Observation
from synphot.models import Empirical1D
import stsynphot as stsyn   # only needed for the ck04models spectra

def load_nircam(name, folder='nircam_throughput'):
    tbl = Table.read(os.path.join(folder, f'{name}.ecsv'))
    return SpectralElement(
        Empirical1D,
        points=tbl['Wavelength'].quantity.to(u.AA),
        lookup_table=tbl['Transmission'].data,
    )

bps = {f: load_nircam(f) for f in ['F115W', 'F150W', 'F200W', 'F356W', 'F444W']}

sp = stsyn.grid_to_spec('ck04models', 5000, -0.5, 4.5)

A_JWST = 2.5e5 * u.cm**2   # ~25 m^2 effective area
for f, bp in bps.items():
    obs = Observation(sp, bp)
    print(f, obs.effstim(flux_unit=u.ABmag), obs.countrate(area=A_JWST))

FileNotFoundError: [Errno 2] No such file or directory: '/grp/redcat/trds/grid/ck04models/catalog.fits'